# 🧠 Kairos CIDA v1.5: Multi-Domain SFT (Agentic, Logic, Math, Creativity) & Extreme TTT Engine

This notebook implements an advanced SFT pipeline and Test-Time Training (TTT) engine targeting general reasoning, agentic planning, and creativity:

1. **Real Multi-Domain Datasets (CoT outputs):**
   - **Math:** GSM8K (Word problems with step-by-step reasoning).
   - **Logic:** ProntoQA (Logical deductions with step-by-step derivation chains).
   - **Agentic Actions:** AgentInstruct (ReAct logs showing multi-turn thoughts and tools).
   - **Creativity:** Riddles v1 (Brainteasers with creative explanations of the solution path).
2. **Selective Parameter Fine-Tuning:** Unfreezes the CIDA attention updater (`cida_core.updater`) and the `gate_proj` for training.
3. **LLM Teacher Evaluator:** Implements a robust LLM-based grader to evaluate generated answers against reference reasoning paths.
4. **Test-Time Training (TTT) on Weights:** Adapts CIDA's attention weights at inference time for a specific prompt to minimize disagreement.

In [ ]:
# Install dependencies
try:
    import bitsandbytes
    import accelerate
except ImportError:
    !pip install -q bitsandbytes accelerate datasets==2.19.0 transformers huggingface_hub tqdm matplotlib

import getpass, sys, os
token = getpass.getpass("GitHub PAT: ").strip()
if token:
    !git clone https://{token}@github.com/Kairatzh/CIDA-plugin.git
    sys.path.append("CIDA-plugin")
    !pip install -q -e CIDA-plugin
else:
    !git clone https://github.com/Kairatzh/CIDA-plugin.git
    sys.path.append("CIDA-plugin")
    !pip install -q -e CIDA-plugin

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
import os, gc, torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from cida_plugin.config import CIDAPluginConfig
from cida_plugin.core import CIDAPlugin

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen1.5-7B-Chat"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_8bit=True, device_map="auto")

print("Loading Base LLM (7B) in 8-bit mode...")
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    quantization_config=bnb_config,
    trust_remote_code=True
)
for p in llm.parameters():
    p.requires_grad = False
llm.eval()

print("Downloading CIDA v1.0 weights from HuggingFace...")
hf_repo = "kirtzh/Kairos-7B-CIDA-v1.0"
cfg_path = hf_hub_download(repo_id=hf_repo, filename="config.json")
cida_bin = hf_hub_download(repo_id=hf_repo, filename="pytorch_model.bin")
proj_bin = hf_hub_download(repo_id=hf_repo, filename="plan_projector.pt")

cfg = CIDAPluginConfig.from_pretrained(os.path.dirname(cfg_path))
cida_core = CIDAPlugin(cfg).to(DEVICE).float()
cida_core.load_state_dict(torch.load(cida_bin, map_location=DEVICE))
print("✅ CIDA v1.0 weights loaded.")

class CIDAPlanProjectorOld(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=8):
        super().__init__()
        self.num_plan_tokens = num_plan_tokens
        self.d_llm = d_llm
        d_flat = num_agents * d_cida
        self.projector = nn.Sequential(
            nn.Linear(d_flat, d_flat*2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d_flat*2, num_plan_tokens*d_llm),
            nn.LayerNorm(num_plan_tokens*d_llm))
    def forward(self, state_s):
        B = state_s.size(0)
        return self.projector(state_s.reshape(B,-1).float()).view(B, self.num_plan_tokens, self.d_llm)

base_proj = CIDAPlanProjectorOld(cfg.d_hidden, llm.config.hidden_size, cfg.num_agents, 8).to(DEVICE)
base_proj.load_state_dict(torch.load(proj_bin, map_location=DEVICE))
print("✅ CIDA v1.0 Old Projector weights loaded.")

class LLMAgnosticCIDAWrapper(nn.Module):
    def __init__(self, cida_core, base_projector, d_llm):
        super().__init__()
        self.cida_core = cida_core
        self.base_projector = base_projector
        self.d_llm = d_llm

        # Gating module
        self.gate_proj = nn.Linear(d_llm * 2, 1)
        nn.init.zeros_(self.gate_proj.weight)
        nn.init.constant_(self.gate_proj.bias, 0.5)

        self.last_rounds_used = 0

    def forward(self, h_base, alpha=2.2):
        co = self.cida_core(pooled=h_base)
        self.last_rounds_used = co["rounds_used"]

        plan = self.base_projector(co["state_final"].s)
        plan_pooled = plan.mean(dim=1)

        gate_in = torch.cat([h_base, plan_pooled], dim=-1).to(self.gate_proj.weight.dtype)
        gate = torch.sigmoid(self.gate_proj(gate_in)).unsqueeze(1)

        h_base_exp = h_base.unsqueeze(1).expand(-1, plan.size(1), -1)
        mixed_plan = gate * (plan * alpha) + (1.0 - gate) * h_base_exp
        return mixed_plan

proj = LLMAgnosticCIDAWrapper(cida_core, base_proj, llm.config.hidden_size).to(DEVICE)

# Configure unfreezing: Only gate_proj and cida_core.updater (attention updating logic)
for name, p in proj.named_parameters():
    if "gate_proj" in name or "cida_core.updater" in name:
        p.requires_grad = True
    else:
        p.requires_grad = False

print("❄️ Set parameter states: gate_proj & cida_core.updater are Trainable. Projectors remain Frozen.")

tuned_weights_path = "Kairos_CIDA_v1.5_FineTuned/pytorch_model.bin"
if os.path.exists(tuned_weights_path):
    proj.load_state_dict(torch.load(tuned_weights_path, map_location=DEVICE))
    print("✅ Fine-tuned CIDA v1.5 weights loaded successfully from disk.")
else:
    print("🚀 Decoupled LLMAgnosticCIDAWrapper constructed successfully (untrained).")

In [ ]:
import random
from datasets import load_dataset

print("Loading real reasoning, agentic, and creative datasets...")

mixed_data = []

# 1. Math: GSM8K (Word Math Problems with step-by-step reasoning solutions)
try:
    gsm = load_dataset("openai/gsm8k", "main", split="train", trust_remote_code=True)
    count = 0
    for ex in list(gsm):
        if count >= 1500: break
        if len(ex["answer"]) < 400:
            mixed_data.append({
                "question": ex["question"],
                "answer": ex["answer"],
                "category": "math"
            })
            count += 1
    print(f"Loaded {count} Math tasks from GSM8K.")
except Exception as e:
    print("Failed to load GSM8K:", e)

# 2. Logic: ProntoQA (Logic deduction problems with detailed step-by-step derivations)
try:
    pronto = load_dataset("longface/prontoqa-train", split="train", trust_remote_code=True)
    count = 0
    for ex in list(pronto):
        if count >= 1500: break
        text = ex["prompt"]
        if "###Response:" in text:
            parts = text.split("###Response:")
            q = parts[0].replace("###Context:", "").strip()
            a = parts[1].strip()
            mixed_data.append({
                "question": q,
                "answer": a,
                "category": "logic"
            })
            count += 1
    print(f"Loaded {count} Logic tasks from ProntoQA.")
except Exception as e:
    print("Failed to load ProntoQA:", e)

# 3. Agentic Actions: AgentInstruct (ReAct logs of OS, Database, and WebShop commands)
try:
    agent_count = 0
    for split_name in ['os', 'db', 'webshop']:
        try:
            d = load_dataset('zai-org/AgentInstruct', split=split_name, trust_remote_code=True)
            count = 0
            for item in list(d):
                if count >= 500: break
                conv = item["conversations"]
                if len(conv) >= 2:
                    q = conv[0]["value"]
                    # Join all assistant turns (bash commands, SQL, thoughts)
                    a_turns = [c["value"] for c in conv if c["from"] == "gpt"]
                    a = "\n\n".join(a_turns)

                    mixed_data.append({
                        "question": q,
                        "answer": a,
                        "category": "agentic"
                    })
                    count += 1
                    agent_count += 1
        except Exception as split_err:
            print(f"Failed to load AgentInstruct split {split_name}:", split_err)
    print(f"Loaded {agent_count} Agentic Action tasks from AgentInstruct.")
except Exception as e:
    print("Failed to load AgentInstruct:", e)

# 4. Creativity: Riddles v1 (Brainteasers with step-by-step explanation of the solution path)
try:
    riddles = load_dataset("Hypersniper/riddles_v1", split="train", trust_remote_code=True)
    count = 0
    for ex in list(riddles):
        if count >= 469: break
        q = ex["instruction"]
        # Target output is the detailed explanation of the creative deduction process
        a = ex["output"]
        mixed_data.append({
            "question": q,
            "answer": a,
            "category": "creativity"
        })
        count += 1
    print(f"Loaded {count} Creative Riddles from Riddles v1.")
except Exception as e:
    print("Failed to load Riddles v1:", e)

random.shuffle(mixed_data)
split_idx = int(len(mixed_data) * 0.9)
train_data = mixed_data[:split_idx]
eval_data = mixed_data[split_idx:]

print(f"Total dataset split -> Train: {len(train_data)} | Held-out Eval: {len(eval_data)}")

In [ ]:
import torch.nn.functional as F
import torch.optim as optim

proj.cida_core.config.early_stop_threshold = None

# We optimize gate_proj with a normal learning rate, and the updater with a very low learning rate
# to slowly adapt the logic engine without destroying pre-trained capabilities.
optimizer = optim.AdamW([
    {"params": proj.gate_proj.parameters(), "lr": 1e-4},
    {"params": proj.cida_core.updater.parameters(), "lr": 1e-5}
], weight_decay=1e-4)

proj.train()

print("Starting Multi-Domain SFT Training (1 Epoch)...")

for idx, sample in enumerate(train_data):
    q = sample["question"]
    ans = sample["answer"]

    prompt = f"<|im_start|>system\nSolve logically with a step-by-step reasoning plan.\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"

    p_enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    a_enc = tokenizer(ans, return_tensors="pt", truncation=True, max_length=384).to(DEVICE)

    with torch.no_grad():
        out = llm(input_ids=p_enc.input_ids, attention_mask=p_enc.attention_mask, output_hidden_states=True)
        compute_dtype = out.hidden_states[-1].dtype
        h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
        p_emb = llm.get_input_embeddings()(p_enc.input_ids).detach()
        a_emb = llm.get_input_embeddings()(a_enc.input_ids).detach()

    plan = proj(h_base)

    full_emb = torch.cat([p_emb.to(compute_dtype), plan.to(compute_dtype), a_emb.to(compute_dtype)], dim=1)
    plan_mask = torch.ones(1, 8, dtype=p_enc.attention_mask.dtype, device=DEVICE)
    full_mask = torch.cat([p_enc.attention_mask, plan_mask, a_enc.attention_mask], dim=1)

    lm_out = llm(inputs_embeds=full_emb, attention_mask=full_mask)
    logits = lm_out.logits.float()

    answer_start = p_emb.size(1) + 8
    La = a_enc.input_ids.size(1)
    answer_logits = logits[0, answer_start - 1 : answer_start + La - 1, :]
    answer_targets = a_enc.input_ids[0]
    loss_mask = a_enc.attention_mask[0].bool()

    loss_per_tok = F.cross_entropy(answer_logits, answer_targets, reduction="none")
    ce_loss = (loss_per_tok * loss_mask.float()).sum() / loss_mask.float().sum().clamp(min=1)

    optimizer.zero_grad()
    ce_loss.backward()
    torch.nn.utils.clip_grad_norm_(proj.parameters(), 1.0)
    optimizer.step()

    if (idx + 1) % 50 == 0:
        print(f"Step {idx+1}/{len(train_data)} | Loss: {ce_loss.item():.4f}")

    del out, p_emb, a_emb, full_emb, lm_out, logits
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
print("Starting Held-out Validation...")
proj.eval()
proj.cida_core.config.early_stop_threshold = None

# Save the fine-tuned checkpoint
save_dir = "Kairos_CIDA_v1.5_FineTuned"
os.makedirs(save_dir, exist_ok=True)
torch.save(proj.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
cfg.save_pretrained(save_dir)
print("Saved modular decoupled weights successfully.")

In [ ]:
save_dir = "Kairos_CIDA_v1.5_FineTuned"
os.makedirs(save_dir, exist_ok=True)
if not os.path.exists(os.path.join(save_dir, "config.json")) and 'cfg' in globals():
    cfg.save_pretrained(save_dir)

try:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    drive_path = "/content/drive/MyDrive/Kairos_CIDA_v1.5_FineTuned"
    os.makedirs(drive_path, exist_ok=True)
    shutil.copy2("Kairos_CIDA_v1.5_FineTuned/pytorch_model.bin", drive_path)
    shutil.copy2("Kairos_CIDA_v1.5_FineTuned/config.json", drive_path)
    print(f"✅ Successfully backed up to Google Drive: {drive_path}")
except Exception as e:
    print(f"Google Drive option skipped or failed: {e}")

In [ ]:
import torch
import torch.nn as nn
import time

print("Initializing original baseline CIDA v1.0 weights for comparison...")

cida_baseline = CIDAPlugin(cfg).to(DEVICE).float()
cida_baseline.load_state_dict(torch.load(cida_bin, map_location=DEVICE))
cida_baseline.eval()

proj_baseline = CIDAPlanProjectorOld(cfg.d_hidden, llm.config.hidden_size, cfg.num_agents, 8).to(DEVICE)
proj_baseline.load_state_dict(torch.load(proj_bin, map_location=DEVICE))
proj_baseline.eval()

proj.eval()
proj.cida_core.config.early_stop_threshold = None
cida_baseline.config.early_stop_threshold = None

# Select benchmark items from held-out evaluation dataset
benchmark_tasks = []
for cat in ["math", "logic", "agentic", "creativity"]:
    items = [s for s in eval_data if s["category"] == cat][:10]
    benchmark_tasks.extend(items)

print(f"Starting comparison benchmark on {len(benchmark_tasks)} multi-domain reasoning tasks...")

# Grader prompt for Qwen model to act as a strict objective teacher
def grade_answer_with_llm_teacher(llm, tokenizer, question, reference, student):
    grader_prompt = (
        "<|im_start|>system\n"
        "You are a strict, objective academic grader. Compare the student's answer with the reference answer to determine if the student's logic is correct and reaches the correct final conclusion.\n"
        "Respond with exactly 'CORRECT' or 'INCORRECT'. Do not write any other explanation or text.\n"
        "<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Question: {question}\n"
        f"Reference Answer: {reference}\n"
        f"Student Answer: {student}\n\n"
        f"Is the student's answer correct?\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    enc = tokenizer(grader_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        ids = llm.generate(**enc, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        gen_text = tokenizer.decode(ids[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip().upper()
    return "CORRECT" in gen_text

results = {
    "v1.0": {"math_ok": 0, "logic_ok": 0, "agentic_ok": 0, "creativity_ok": 0, "tokens": 0, "latency": 0.0},
    "v1.5": {"math_ok": 0, "logic_ok": 0, "agentic_ok": 0, "creativity_ok": 0, "tokens": 0, "latency": 0.0},
    "cot": {"math_ok": 0, "logic_ok": 0, "agentic_ok": 0, "creativity_ok": 0, "tokens": 0, "latency": 0.0}
}

# Run evaluation
for idx, s in enumerate(benchmark_tasks):
    q, correct, cat = s["question"], s["answer"], s["category"]

    prompt = f"<|im_start|>system\nSolve logically.\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out = llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
        compute_dtype = out.hidden_states[-1].dtype
        h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
        p_emb = llm.get_input_embeddings()(enc.input_ids).detach()
        gm = torch.cat([enc.attention_mask, torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=DEVICE)], dim=1)

        # 1. CIDA v1.0 Baseline
        t0 = time.time()
        co_old = cida_baseline(pooled=h_base)
        plan_old = proj_baseline(co_old["state_final"].s)
        ids_old = llm.generate(inputs_embeds=torch.cat([p_emb.to(compute_dtype), plan_old.to(compute_dtype)], dim=1), attention_mask=gm, max_new_tokens=150)
        text_old = tokenizer.decode(ids_old[0], skip_special_tokens=True).strip()
        dt_old = time.time() - t0

        # 2. CIDA v1.5 Gated
        t0 = time.time()
        plan_new = proj(h_base)
        ids_new = llm.generate(inputs_embeds=torch.cat([p_emb.to(compute_dtype), plan_new.to(compute_dtype)], dim=1), attention_mask=gm, max_new_tokens=150)
        text_new = tokenizer.decode(ids_new[0], skip_special_tokens=True).strip()
        dt_new = time.time() - t0

        # 3. Chain-of-Thought (CoT)
        cot_prompt = f"<|im_start|>system\nSolve step-by-step with explanation.\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
        enc_cot = tokenizer(cot_prompt, return_tensors="pt").to(DEVICE)
        t0 = time.time()
        ids_cot = llm.generate(**enc_cot, max_new_tokens=300)
        text_cot = tokenizer.decode(ids_cot[0][enc_cot.input_ids.shape[1]:], skip_special_tokens=True).strip()
        dt_cot = time.time() - t0

        # Log metrics
        results["v1.0"]["tokens"] += len(ids_old[0])
        results["v1.0"]["latency"] += dt_old
        results["v1.5"]["tokens"] += len(ids_new[0])
        results["v1.5"]["latency"] += dt_new
        results["cot"]["tokens"] += len(ids_cot[0]) - enc_cot.input_ids.shape[1]
        results["cot"]["latency"] += dt_cot

        # Semantic Grading via LLM Teacher
        ok_old = grade_answer_with_llm_teacher(llm, tokenizer, q, correct, text_old)
        ok_new = grade_answer_with_llm_teacher(llm, tokenizer, q, correct, text_new)
        ok_cot = grade_answer_with_llm_teacher(llm, tokenizer, q, correct, text_cot)

        cat_key = f"{cat}_ok"
        if ok_old: results["v1.0"][cat_key] += 1
        if ok_new: results["v1.5"][cat_key] += 1
        if ok_cot: results["cot"][cat_key] += 1

flops_per_tok = 14.0 * 1e9
flops_cida_deliberation = 3.0 * 40.0 * 1e6

flops_old = (results["v1.0"]["tokens"] * flops_per_tok) + (len(benchmark_tasks) * flops_cida_deliberation)
flops_new = (results["v1.5"]["tokens"] * flops_per_tok) + (len(benchmark_tasks) * flops_cida_deliberation)
flops_cot = results["cot"]["tokens"] * flops_per_tok

speedup = flops_cot / flops_new
compute_savings = (1 - (flops_new / flops_cot)) * 100

print("\n=========================================================================")
print("📈 KAIROS CIDA MULTI-DOMAIN ADVANTAGE AUDIT REPORT")
print("=========================================================================")
print(f"| Metric                      | CoT (Standard) | CIDA v1.0 (Old) | CIDA v1.5 (Gated) |")
print(f"|-----------------------------|----------------|-----------------|-------------------|")
print(f"| Math Accuracy (%)           | {results['cot']['math_ok']/10*100:13.1f}% | {results['v1.0']['math_ok']/10*100:14.1f}% | {results['v1.5']['math_ok']/10*100:16.1f}% |")
print(f"| Logical Accuracy (%)        | {results['cot']['logic_ok']/10*100:13.1f}% | {results['v1.0']['logic_ok']/10*100:14.1f}% | {results['v1.5']['logic_ok']/10*100:16.1f}% |")
print(f"| Agentic Action Accuracy (%) | {results['cot']['agentic_ok']/10*100:13.1f}% | {results['v1.0']['agentic_ok']/10*100:14.1f}% | {results['v1.5']['agentic_ok']/10*100:16.1f}% |")
print(f"| Creative Riddle Accuracy (%)| {results['cot']['creativity_ok']/10*100:13.1f}% | {results['v1.0']['creativity_ok']/10*100:14.1f}% | {results['v1.5']['creativity_ok']/10*100:16.1f}% |")
print(f"| Total Generated Tokens      | {results['cot']['tokens']:14d} | {results['v1.0']['tokens']:15d} | {results['v1.5']['tokens']:17d} |")
print(f"| Cumulative Latency (sec)    | {results['cot']['latency']:13.2f}s | {results['v1.0']['latency']:14.2f}s | {results['v1.5']['latency']:16.2f}s |")
print(f"| Estimated FLOPs Cost        | {flops_cot/1e12:12.3f}T | {flops_old/1e12:13.3f}T | {flops_new/1e12:15.3f}T |")
print("=========================================================================")
print(f"💡 CIDA v1.5 Gating vs standard CoT Speedup: {speedup:.2f}x")
print(f"💡 CIDA v1.5 Computational/FLOPs Savings: {compute_savings:.2f}%")
print("=========================================================================\n")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

class KairosExtremeReasoningEngine(nn.Module):
    """
    An advanced test-time search and optimization engine.
    It runs:
    1. Latent MCTS path exploration (generating multiple alternative plan anchors).
    2. Test-Time Training (TTT): dynamically adjusts the CIDA Attention weights 
       and the plan embeddings during the forward pass to minimize agent disagreement.
    3. Dynamic Unlimited rounds: runs agent updates until consensus stabilizes.
    """
    def __init__(self, wrapper_proj, llm, tokenizer):
        super().__init__()
        self.proj = wrapper_proj
        self.llm = llm
        self.tokenizer = tokenizer
        self.device = next(wrapper_proj.parameters()).device

    def solve_extreme(self, prompt: str, num_alternatives: int = 5, tto_steps: int = 15, max_dynamic_rounds: int = 15):
        print(f"\n[TTO Engine] Processing prompt: \"{prompt[:60]}...\"")
        enc = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        with torch.no_grad():
            out = self.llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
            compute_dtype = out.hidden_states[-1].dtype
            h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
            p_emb = self.llm.get_input_embeddings()(enc.input_ids).detach()

        # Save a backup of the original CIDA Attention Updater weights
        updater_backup = {k: v.clone() for k, v in self.proj.cida_core.updater.state_dict().items()}

        # --- 1. MCTS Branch Exploration & TTT Weight Adaptation ---
        best_plan = None
        best_score = float("inf")
        best_co = None
        best_updater_weights = None

        print(f"[TTO Engine] Spawning {num_alternatives} alternative latent paths (MCTS Nodes)...")

        # Temporarily set max_rounds in CIDA core to max_dynamic_rounds
        orig_max_rounds = self.proj.cida_core.config.max_rounds
        self.proj.cida_core.config.max_rounds = max_dynamic_rounds

        for path_idx in range(num_alternatives):
            # Restore original weights before starting this path's training
            self.proj.cida_core.updater.load_state_dict(updater_backup)
            self.proj.cida_core.updater.requires_grad_(True)

            with torch.no_grad():
                base_co = self.proj.cida_core(pooled=h_base)
                anchor_plan = self.proj.base_projector(base_co["state_final"].s)

            if path_idx == 0:
                alpha = 2.2
                plan_candidate = anchor_plan * alpha
            elif path_idx == 1:
                alpha = 2.8
                plan_candidate = anchor_plan * alpha
            elif path_idx == 2:
                alpha = 1.5
                plan_candidate = anchor_plan * alpha
            else:
                # Explore alternative coordinate space with minor random perturbation
                alpha = 2.2
                noise = torch.randn_like(anchor_plan) * 0.05
                plan_candidate = (anchor_plan + noise) * alpha

            # Make candidate trainable for TTO
            plan_opt = plan_candidate.clone().detach().requires_grad_(True)

            # Optimize both the plan embedding AND the CIDA attention updater parameters!
            ttt_optimizer = optim.Adam(
                list(self.proj.cida_core.updater.parameters()) + [plan_opt], 
                lr=0.01
            )

            # Run Test-Time Training (TTT) steps
            for step in range(tto_steps):
                ttt_optimizer.zero_grad()

                # Forward pass through CIDA core (with current updated attention parameters)
                co = self.proj.cida_core(pooled=h_base, seq_output=plan_opt)

                # Target: minimize agent disagreement + uncertainty
                loss = co["disagreement"].mean() + co["uncertainty"].mean()

                loss.backward()
                ttt_optimizer.step()

            # Evaluate final consensus score of the optimized branch
            with torch.no_grad():
                final_co = self.proj.cida_core(pooled=h_base, seq_output=plan_opt)
                score = final_co["disagreement"].item() + final_co["uncertainty"].item()

            # Track best alternative path and its corresponding optimized CIDA updater weights
            if score < best_score:
                best_score = score
                best_plan = plan_opt.detach()
                best_co = final_co
                best_updater_weights = {k: v.clone() for k, v in self.proj.cida_core.updater.state_dict().items()}

            print(f"  -> Path {path_idx} TTT Score (Disagreement+Uncertainty): {score:.4f} (Rounds used: {final_co['rounds_used']})")

        # Restore CIDA config
        self.proj.cida_core.config.max_rounds = orig_max_rounds

        # Load the best performing CIDA updater weights for the final generation
        self.proj.cida_core.updater.load_state_dict(best_updater_weights)
        self.proj.cida_core.updater.requires_grad_(False)

        # --- 2. Generation on Best Optimized Plan ---
        print(f"[TTO Engine] Best TTT path selected with score {best_score:.4f}. Generating answer...")
        t0 = time.time()
        gm = torch.cat([enc.attention_mask, torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=self.device)], dim=1)

        with torch.no_grad():
            ids = self.llm.generate(
                inputs_embeds=torch.cat([p_emb.to(compute_dtype), best_plan.to(compute_dtype)], dim=1),
                attention_mask=gm,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
            text = self.tokenizer.decode(ids[0], skip_special_tokens=True).strip()
        dt = time.time() - t0

        # Crucial: Reset the CIDA updater weights back to original for subsequent calls
        self.proj.cida_core.updater.load_state_dict(updater_backup)

        return {
            "text": text,
            "score": best_score,
            "rounds": best_co["rounds_used"],
            "disagreement": best_co["disagreement"].item(),
            "uncertainty": best_co["uncertainty"].item(),
            "latency": dt
        }

# Initialize the engine
engine = KairosExtremeReasoningEngine(proj, llm, tokenizer)

# ── Test Case: Hard Einstein's Riddle Logic Puzzle ──
hard_prompt = (
    "<|im_start|>system\nYou are a logical detective. Solve step by step.\n<|im_end|>\n"
    "<|im_start|>user\n"
    "Three houses are in a row: Red, Blue, and Green. Each house has a different owner: Anton, Boris, and Claus. "
    "Each owner has a different pet: Dog, Cat, and Zebra. Here are the rules:\n"
    "1. The Blue house is in the middle.\n"
    "2. Anton owns the Red house.\n"
    "3. The person in the Green house owns a Cat.\n"
    "4. Boris owns a Dog.\n\n"
    "Question: Who owns the Zebra? Explain your deduction step by step.\n"
    "<|im_end|>\n<|im_start|>assistant\n"
)

# Run standard generation (no TTO)
enc = tokenizer(hard_prompt, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
    compute_dtype = out.hidden_states[-1].dtype
    h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
    p_emb = llm.get_input_embeddings()(enc.input_ids).detach()
    gm = torch.cat([enc.attention_mask, torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=DEVICE)], dim=1)

    # Standard Plan (CIDA 1.5)
    p_std = proj(h_base)
    ids_std = llm.generate(inputs_embeds=torch.cat([p_emb.to(compute_dtype), p_std.to(compute_dtype)], dim=1), attention_mask=gm, max_new_tokens=200)
    text_std = tokenizer.decode(ids_std[0], skip_special_tokens=True).strip()

# Run Extreme Engine (MCTS + TTT weight adaptation)
ttt_res = engine.solve_extreme(hard_prompt, num_alternatives=5, tto_steps=15, max_dynamic_rounds=15)

print("\n========================================================")
# Print comparison
print("🧠 INFERENCE COMPARISON ON EINSTEIN'S RIDDLE (WITH TTT)")
print("========================================================")
print("--- CIDA v1.5 Standard Output ---")
print(text_std)
print("\n--- CIDA Extreme (TTT Weight Adaptation) Output ---")
print(ttt_res["text"])
print(f"\n⏱️ Latency: {ttt_res['latency']:.2f}s | 🔄 Dynamic Rounds: {ttt_res['rounds']} | 🛡️ Final Disagreement: {ttt_res['disagreement']:.4f}")
print("========================================================\n")